# Recorrido completo del código — dinámica no local *point-splitting*Implementación de J. Polonyi, [arXiv:1701.04068v4](https://arxiv.org/abs/1701.04068).Este notebook recorre el paquete `nlaid/` **de abajo hacia arriba**, siguiendo elgrafo de dependencias. Cada sección verifica lo que la anterior dejó establecido,de modo que si algo falla sabes exactamente dónde.```core.py  ────────────────────────────────  base, no importa nada   │   ├── block1_linear.py                    RAMA ESPECTRAL   (ec. 14)   │   ├── worldline.py                        RAMA TEMPORAL   │      ├── block2_delay.py              (ec. 16)   │      └── block3_memory.py             (ec. 17)   │   └── block4_renorm.py  ←  junta las dos ramas```Las dos ramas son **independientes**: `block1` no toca `worldline`, y `block2/3`no tocan `block1`. Solo comparten `core`. Esa independencia es lo que hace que laprueba cruzada de la sección 6 signifique algo.**Costo**: las secciones 0–3 son instantáneas. Las 4–6 hacen integraciones desegundos a minutos. Los barridos completos **no se recalculan aquí**: se cargande `figures/*.npz` y `data/*.json`.

In [ ]:
import sys, pathlib, jsonimport numpy as npimport matplotlib.pyplot as plt# Ejecutar desde la raíz del repo (o ajustar esta ruta)RAIZ = pathlib.Path.cwd()if not (RAIZ / "nlaid").is_dir():    RAIZ = RAIZ.parentsys.path.insert(0, str(RAIZ))np.set_printoptions(precision=6, suppress=False)plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True,                     "grid.alpha": .3, "font.size": 10})print("raíz:", RAIZ)

## 0. ConvencionesNada de lo que sigue se lee bien sin esto:| | ||---|---|| Signatura | $(+,-,-,-)$ — las líneas de mundo temporales tienen $x^2>0$ || Unidades | $c=1$, $r_0=1$: el cutoff $\ell$ se mide en unidades del radio clásico || Parametrización | tiempo propio $s$, con $\dot x^2 = 1$ || Dimensión | `dim=2` (1+1) por defecto, que es donde el paper hace sus figuras |Los dos parámetros libres del modelo son exactamente los ejes de la Fig. 2 delpaper: $r_0/\ell$ y $m/m_B$.

In [ ]:
from nlaid.core import Params, minkowski_dot, make_regulatorp = Params(ell=1/3, m_over_mB=2.0)     # r0/ell = 3, el caso de la Fig. 1print(p)print("r0/ell =", p.r0_over_ell)# La métrica: un vector temporal unitario tiene norma +1v = np.array([np.cosh(0.7), np.sinh(0.7)])print("xdot^2 =", minkowski_dot(v, v))

## 1. `core.py` — reguladores y momentosEl regulador es **la no-localidad**. Reemplaza la delta de la función de Greenretardada, $\delta(x^2)\to\delta_B(x^2)$, sujeta a tres condiciones (p. 6 del paper):| # | Condición | Razón física ||---|---|---|| (i) | $\int dz\,\delta_B(z)=1$ | preservar el flujo del campo radiado || (ii) | $\delta_B(0)=0$ | separar los puntos singulares || (iii) | $\delta_B(z)=0$ para $z<0$ | **suprimir la interacción superlumínica** |La (iii) es el mecanismo entero del paper: una trayectoria *runaway* tendría quesuperar $c$ para escapar, y ahí el regulador apaga la autointeracción.Dos realizaciones: **ec. (4)** desplazada, $\delta_B(x^2)=\delta(x^2-\ell^2)$, y**ec. (5)** suavizada, $\delta_B(x^2)=\Theta(x^2)\,x^2 e^{-\sqrt{x^2}/\ell}/(12\ell^4)$.

In [ ]:
ell = 0.4suav = make_regulator("smeared", ell)desp = make_regulator("shifted", ell)print("condiciones del regulador suavizado:")for k, v in suav.check_conditions().items():    print(f"   {k:22s} {v}")z = np.linspace(-0.2, 6*ell**2, 600)plt.plot(z, suav.delta(z), lw=2, label=r"$\delta_B(z)$, ec. (5)")plt.axvline(0, color="k", lw=.8)plt.xlabel("$z = x^2$"); plt.ylabel(r"$\delta_B$")plt.title(f"Regulador suavizado, $\\ell$ = {ell}")plt.legend(); plt.show()

### Momentos contra las formas cerradasEstas expresiones **no están en el paper**; las derivé para poder validar elcódigo contra valores exactos en lugar de contra otra corrida del propio código.| Cantidad | Suavizado, ec. (5) | Desplazado, ec. (4) ||---|---|---|| $\int_0^\infty dz\, z^{-1/2}\delta_B(z)$ | $1/(3\ell)$ | $1/\ell$ || $\delta m/m$ vía ec. (10) | $r_0/(6\ell)$ | $r_0/(2\ell)$ || $I_2=\int_{-\infty}^0 du\,\delta_B(u^2)u^2$ | $2\ell$ | $\ell/2$ |**Errata detectada al transcribir**: los extractores automáticos de texto leen laec. (10) como $\int dz\sqrt{z}\,\delta_B(z)$, dimensionalmente imposible. La imagende la p. 7 muestra $\int dz/\sqrt{z}\,\delta_B(z)$.

In [ ]:
print(f"{'':12s} {'delta_m/m':>12} {'esperado':>12} | {'I2':>10} {'esperado':>10}")for nom, reg, dm_ex, i2_ex in (("suavizado", suav, 1/(6*ell), 2*ell),                               ("desplazado", desp, 1/(2*ell), ell/2)):    print(f"{nom:12s} {reg.mass_shift_over_m():12.8f} {dm_ex:12.8f} | "          f"{reg.moment_u2():10.6f} {i2_ex:10.6f}")print("\nm_B cambia de signo cuando delta_m = m:")for kind, x_c in (("desplazado", 2.0), ("suavizado", 6.0)):    print(f"   {kind:11s} en r0/ell = {x_c}")

## 2. `block1_linear.py` — la teoría linealizada, ec. (14)Autocontenido: solo depende de `core`. Calcula la susceptibilidad$$\chi^r_\omega = 1 + r_0\left[\tfrac23 i\omega - \frac{2}{\omega^2}\int_{-\infty}^{0}du\,\delta_B(u^2)\,\frac{N(\omega u)}{u^2}\right]$$con $N(\phi)=(1+i\phi-\phi^2)e^{-i\phi}-1+\tfrac12\phi^2-\tfrac23 i\phi^3$.Como $F^r_\omega = 1/[(\omega+i\epsilon)^2\chi^r_\omega]$, **los polos de $F^r$ son losceros de $\chi^r$**. La dinámica es estable y causal si no hay ceros en$\mathrm{Im}\,\omega>0$.

### 2.1 Por qué existe la serie de $N(\phi)$Los cuatro primeros órdenes de $N$ se cancelan **idénticamente** con la sustracción,dejando$$N(\phi)=\sum_{n\ge4}\frac{(-i\phi)^n (n-1)^2}{n!}$$(se comprueba con $\sum (n-1)^2x^n/n! = (x^2-x+1)e^x$ y $x=-i\phi$). El ordendominante es $\tfrac38\phi^4$, que **coincide exactamente** con lo que afirma elpaper en la p. 8 — ésa es la verificación de que la ec. (14) está bien transcrita.La celda siguiente muestra por qué no se puede evaluar $N$ con la forma directa:a $\phi$ pequeño la cancelación destruye toda la precisión.

In [ ]:
from nlaid.block1_linear import numerator_Nphis = np.logspace(-6, -1, 40)directa = ((1 + 1j*phis - phis**2)*np.exp(-1j*phis) - 1           + 0.5*phis**2 - (2/3)*1j*phis**3)exacto = (3/8)*phis**4                      # orden dominanteserie = np.array([numerator_N(f).real for f in phis])plt.loglog(phis, np.abs(directa.real - exacto)/exacto, "o-", label="forma directa")plt.loglog(phis, np.abs(serie - exacto)/np.maximum(exacto, 1e-300) + 1e-17,           "s-", label="serie (el código)")plt.xlabel(r"$\phi$"); plt.ylabel("error relativo vs $(3/8)\\phi^4$")plt.title("Cancelación catastrófica en el numerador de la ec. (14)")plt.legend(); plt.show()

### 2.2 La susceptibilidad y sus dos controles- **Desarrollo a $\omega$ pequeño**: $\chi = 1 + r_0[\tfrac23 i\omega - \tfrac34\omega^2 I_2]+O(\omega^3)$.- **Forma cerrada** para el regulador desplazado: como  $\int_{-\infty}^0 du\,\delta_B(u^2)f(u) = f(-\ell)/2\ell$, se obtiene  $\chi = 1 + r_0[\tfrac23 i\omega - N(-\omega\ell)/(\omega^2\ell^3)]$ **exacta**.  Es el test más fuerte de la cuadratura del caso suavizado.**Validez**: sustituyendo $u=-v$, el integrando decae como$v^2e^{-v(1/\ell+\mathrm{Im}\,\omega)}$, luego converge para$\mathrm{Im}\,\omega > -1/\ell$ — **todo el semiplano superior**. El"requiere $\mathrm{Im}\,\omega>0$" del paper es condición suficiente, no el límite.

In [ ]:
from nlaid.block1_linear import susceptibility, susceptibility_small_omegape = Params(ell=0.4)w = np.linspace(0.01, 3, 300)chi = susceptibility(w, suav, pe)fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))ax[0].plot(w, chi.real, label="Re"); ax[0].plot(w, chi.imag, label="Im")ax[0].set_xlabel(r"$\omega$"); ax[0].set_title(r"$\chi^r_\omega$, suavizado"); ax[0].legend()ws = np.linspace(1e-3, .3, 80)ax[1].plot(ws, np.abs(susceptibility(ws, suav, pe)                      - susceptibility_small_omega(ws, suav, pe)), lw=2)ax[1].set_xscale("log"); ax[1].set_yscale("log")ax[1].set_xlabel(r"$\omega$"); ax[1].set_title(r"$|\chi - \chi_{\rm serie}|$  (debe ir como $\omega^3$)")plt.tight_layout(); plt.show()# forma cerrada del regulador desplazadowt = np.array([0.2, 1.0, 3.0, 8.0 + 0.3j])cerrada = 1 + pe.r0*((2/3)*1j*wt - numerator_N(-wt*pe.ell)/(wt**2*pe.ell**3))print("max |numérico - cerrado| =",      np.max(np.abs(susceptibility(wt, desp, pe) - cerrada)))

### 2.3 Los ceros: el polo runaway de Abraham-LorentzAl remover el cutoff, $\chi^r_\omega \to 1+\tfrac23 i r_0\omega$ (ec. 15), cuyo ceroestá en $\omega = 3i/2$ — **semiplano superior**: la autoaceleración.El conteo se hace por **principio del argumento** (número de vueltas de $\chi$alrededor del origen sobre un contorno cerrado), que es global y no depende desemillas, y se contrasta con la localización por **Muller**. Que ambos coincidanes el control de que no se pierde ni se inventa ningún cero.

In [ ]:
from nlaid.block1_linear import count_zeros_uhp, find_zeros_uhpell_p = 0.02                                  # cutoff muy fino: A-L puroregp, pp = make_regulator("smeared", ell_p), Params(ell=ell_p)kw = dict(re_max=6.0, im_hi=6.0)n = count_zeros_uhp(regp, pp, n=600, **kw)zs = find_zeros_uhp(regp, pp, grid=120, **kw)print(f"principio del argumento: {n} ceros      Muller: {len(zs)}")for z in zs:    print(f"   omega = {z.real:+.5f} {z.imag:+.5f}i")print(f"\npredicción de la ec. (15): omega = 3i/2 = {1.5j}")

## 3. `worldline.py` — la historia compartidaEs la pieza común a los bloques 2 y 3: ambos necesitan lo mismo — consultar$x,\dot x,\ddot x$ en un tiempo propio pasado arbitrario.**Condición inicial.** Una carga en reposo es solución **exacta** de las tresecuaciones: con $\dot x=\dot x'=(1,0)$ y $(x-x')$ proporcional a $\dot x$, el vector$V_1=(x-x')(\dot x\dot x')-[\dot x(x-x')]\dot x'$ se anula idénticamente. No hayautofuerza para movimiento inercial, como debe ser.La excitación se aplica con la **fuente externa $k^\mu$ de la ec. (2)** como pulso$C^\infty$ de soporte compacto. Prescribir una trayectoria y apagarla de golpeharía saltar $\ddot x$ en $s=0$; en una ecuación con retardo esa discontinuidad sepropaga a $s=\ell,2\ell,\dots$ y **degrada el orden del integrador de 2 a ~1.2**.

In [ ]:
from nlaid.worldline import rest_history, unit_normal, smooth_bumpwl = rest_history(dim=2, s_rest=10.0, ds=1e-3)print("prehistoria en reposo:  max |xdot^2 - 1| =", wl.norm_drift.max())# El normal unitario preserva la normalización exactamentefor th in (0.0, 0.4, -1.2):    vv = np.array([np.cosh(th), np.sinh(th)]); nn = unit_normal(vv)    print(f"   theta={th:+.1f}   n.v = {minkowski_dot(nn,vv):+.2e}   "          f"n.n = {minkowski_dot(nn,nn):+.6f}")# En reposo el punto retardado está exactamente a distancia ellfor L in (0.1, 0.37, 1.0):    sp, *_ = wl.retarded_point(wl.s_max, wl._x[-1], L)    print(f"   ell={L:4.2f}   retardo s-s' = {wl.s_max - sp:.10f}")

## 4. `block2_delay.py` — ec. (16), retardo finito$$\ddot x = r_0\frac{m}{m_B}\frac{1}{A^2}\left[\frac{B+1}{A}V_1 + V_2\right]$$con $A=\dot x'\!\cdot\!(x-x')$, $B=\ddot x'\!\cdot\!(x'-x)$, y el punto retardadofijado por $\ell^2=(x-x')^2$.**Naturaleza**: DDE de tipo **neutro** con retardo dependiente del estado. $\ddot x'$es historia ya calculada (el retardo está acotado por $\sim\ell$), pero la*ubicación* del punto retardado depende de $x(s)$, que es la incógnita del paso.De ahí el predictor-corrector.### El control más sensible: ortogonalidadContrayendo los lados derechos de las ecs. (6), (16) y (17) con $\dot x_\mu$ seobtiene **cero idénticamente**. Por ejemplo para la (6):$$\dot x\cdot\{(x-x')(\dot x\dot x') - [\dot x(x-x')]\dot x'\} = [\dot x(x-x')](\dot x\dot x') - [\dot x(x-x')](\dot x'\dot x) = 0$$Consecuencia: $\dot x^2=1$ **no es una restricción a imponer sino una consecuenciaexacta**, y su deriva numérica mide el error de integración gratis. Si unatranscripción de la ecuación estuviera mal, este test lo detecta de inmediato.

In [ ]:
from nlaid.block2_delay import rhs_delay, integrate_delayfrom nlaid.block3_memory import rhs_memory, integrate_memorypd = Params(ell=0.3, m_over_mB=0.5)wl2 = integrate_delay(pd, s_end=2.0, ds=2e-3)x, v = wl2._x[-1], wl2._v[-1]_, xp, vp, ap = wl2.retarded_point(wl2.s_max + 1e-9, x, pd.ell)acc2, _ = rhs_delay(x, v, xp, vp, ap, pd)print(f"ec. (16):  xddot . xdot = {minkowski_dot(acc2, v):+.3e}   "      f"|xddot| = {np.linalg.norm(acc2):.4f}")wl3 = integrate_memory(pd, s_end=2.0, ds=2e-3, n_ell=20.0)acc3 = rhs_memory(wl3._x[-1], wl3._v[-1], wl3, wl3.s_max, pd, n_ell=20.0)print(f"ec. (17):  xddot . xdot = {minkowski_dot(acc3, wl3._v[-1]):+.3e}   "      f"|xddot| = {np.linalg.norm(acc3):.4f}")print("\n(cero a precisión de máquina = las ecuaciones están bien transcritas)")

In [ ]:
# Convergencia de orden 2 en ds  (~1 min)import mathprobe = np.linspace(1.5, 4.0, 40)sols = {d: integrate_delay(Params(ell=0.5, m_over_mB=0.4), s_end=4.0, ds=d)        for d in (2e-2, 1e-2, 5e-3, 2.5e-3)}ref = sols[2.5e-3].sample_many(probe)[0]errs = [np.max(np.abs(sols[d].sample_many(probe)[0] - ref)) for d in (2e-2, 1e-2, 5e-3)]print("ds        error        orden")for i, d in enumerate((2e-2, 1e-2, 5e-3)):    o = "" if i == 0 else f"{math.log2(errs[i-1]/errs[i]):.2f}"    print(f"{d:<9.4f} {errs[i]:.3e}   {o}")print("\nderiva de xdot^2:", {d: f"{s.norm_drift.max():.1e}" for d, s in sols.items()})

## 5. `block3_memory.py` — ec. (17), memoria infinita$$\ddot x = 4r_{0B}\int ds'\,\delta_B'\big((x-x')^2\big)\Big\{(x-x')(\dot x\dot x')-[\dot x(x-x')]\dot x'\Big\}$$que es la forma **general** de la ec. (6). El código toma $\delta_B'$ del reguladorde `core`, no reescrito — cambiar el regulador se propaga automáticamente.**Naturaleza**: sistema no lineal de Volterra de **segunda especie** (reducido).Estrictamente es integro-diferencial, pero tomando $u=\dot x$, integrando una vez ycolapsando la doble integral por Fubini, $(x,u)$ satisface un sistema de Volterrade 2da especie. Eso es lo que autoriza la marcha explícita hacia adelante.Tres salvedades: el **núcleo depende de la solución** (no hay resolvente ni seriede Neumann); el límite inferior es $-\infty$ y la prehistoria entra como forzante(por eso es problema de valor inicial, y por eso Polonyi insiste en CTP); y el**núcleo es regular**, no débilmente singular — cuadratura estándar a orden pleno.Hay **tres** discretizaciones que convergen por separado: $ds$, la ventana `n_ell`,y la resolución `pts_per_ell`.

In [ ]:
pm = Params(ell=0.5, m_over_mB=0.4)print("ventana de memoria (pts_per_ell fijo -> mide truncamiento puro):")prev = Nonefor n_ell in (10, 20, 30, 40):    a = np.linalg.norm(integrate_memory(pm, s_end=2.0, ds=1e-2,                                        n_ell=n_ell, pts_per_ell=16)._a[-1])    d = "" if prev is None else f"   cambio = {abs(a-prev):.2e}"    print(f"   n_ell={n_ell:3d}   |xddot| = {a:.12f}{d}")    prev = aprint("\nresolución de la cuadratura:")prev = Nonefor pp_ in (4, 8, 16, 32):    a = np.linalg.norm(integrate_memory(pm, s_end=2.0, ds=1e-2,                                        n_ell=25, pts_per_ell=pp_)._a[-1])    d = "" if prev is None else f"   cambio = {abs(a-prev):.2e}"    print(f"   pts_per_ell={pp_:3d}   |xddot| = {a:.12f}{d}")    prev = a

## 6. `block4_renorm.py` — renormalización y la prueba cruzada### El problema con la ec. (18)$$\chi^r_\omega = 1 + \tfrac23 i r_0\omega \qquad (18)$$Su **único** cero está en $\omega=3i/2$, en el semiplano **superior**. Una teoría quela satisfaga exactamente no tiene ningún modo de relajación que monitorear, así que**no puede imponerse literalmente** sobre una teoría estable. El paper la aplica deforma operacional (p. 10): *"monitoring the relaxation for large $s$"*.Se implementan dos condiciones que sí están bien definidas:- **R1** — tasa no lineal $=\mathrm{Im}\,\omega$ del cero dominante de $\chi^r$.- **R2** — tasa no lineal $=0$: la frontera de la región sombreada de la Fig. 2.### La prueba cruzada`linearized_rate` viene del **bloque 1** (análisis espectral). `relaxation_rate`viene del **bloque 3** (integración temporal no lineal). No comparten código másallá de `core`. Que coincidan valida ambos a la vez.

In [ ]:
from nlaid.block4_renorm import relaxation_rate, linearized_ratepc = Params(ell=1/3, m_over_mB=2.0)          # r0/ell = 3lin = linearized_rate(pc)                     # bloque 1, espectralnolin, diag = relaxation_rate(pc, s_end=14.0, ds=5e-3)   # bloque 3, temporalprint(f"  bloque 1 (espectral)  : {lin:+.5f}")print(f"  bloque 3 (no lineal)  : {nolin:+.5f}")print(f"  diferencia relativa   : {abs(lin-nolin)/abs(lin):.2%}")print(f"  deriva de xdot^2      : {diag['drift']:.1e}   (fiable={diag['fiable']})")print(f"\n  contraterm ec. (10) a r0/ell=3:  m/m_B = "      f"{make_regulator('smeared', 1/3).m_over_mB_counterterm():.4f}")print("  (la Fig. 1a del paper usa 1.95, 1.98 y 2.00 — bracketea ese valor)")

### La ley del borde de estabilidadResultado empírico del barrido R2: la frontera **no** cae a $m/m_B$ fijo sino a$$\frac{r_{0B}}{\ell} = \frac{r_0}{\ell}\cdot\frac{m}{m_B} = \text{constante}$$con $r_{0B}=e^2/(m_Bc^2)$ el radio clásico **desnudo**. El criterio de estabilidadcompara el acoplamiento desnudo con el cutoff, no con $r_0$.

In [ ]:
sm = json.load(open(RAIZ/"data/borde_smeared.json"))print(f"{'r0/ell':>7} {'rama +':>9} {'rama -':>9} | {'x(m/mB)+':>10} {'x(m/mB)-':>10}")for k, v in sorted(sm.items(), key=lambda kv: float(kv[0])):    x = float(k); P, N = v.get("positiva"), v.get("negativa")    f = lambda q: float("nan") if q is None else q    print(f"{x:7.1f} {f(P):9.3f} {f(N):9.3f} | {x*f(P):10.3f} {x*f(N):10.3f}")for rama in ("positiva", "negativa"):    vals = [float(k)*v[rama] for k, v in sm.items() if v.get(rama)]    print(f"\n  r_0B/ell rama {rama:9s}: {np.mean(vals):+8.3f} ± {np.std(vals):.3f}"          f"   ({len(vals)} puntos)")

## 7. Comparación con el paper`data/fig2_digitalizada.npz` contiene la frontera de la Fig. 2 extraída del PDF porpíxeles (calibración en el `.json` acompañante: eje $y=0$ en la fila 446, 367 px porunidad; 59.36 px por unidad de $r_0/\ell$, con la retícula de marcas menoresconfirmada en ambos paneles).**Dos hallazgos**, ambos documentados en `docs/discrepancias.md`:1. La línea punteada de **ambos** paneles sigue $1/(1-r_0/2\ell)$ — el contratérmino   del regulador **desplazado** — pese a que el panel (b) es el suavizado. Aplicando   la ec. (10) a la ec. (5) el resultado correcto es $\delta m = e^2/6c^2\ell$.2. La región del panel (b) es la del panel (a) con la abscisa escalada por   $c=3.03$ (RMS 0.021, frente a 0.617 sin reescalar). Y 3 es exactamente el factor   que iguala los $\delta m$ de ambos reguladores.**Verificar en el PDF original antes de citar cualquiera de las dos curvas.**

In [ ]:
dig = np.load(RAIZ/"data/fig2_digitalizada.npz")fig, axes = plt.subplots(1, 2, figsize=(11.5, 4))for ax, pan, tit in ((axes[0], "a", "Fig. 2(a) — desplazado"),                     (axes[1], "b", "Fig. 2(b) — suavizado")):    xd, sup, inf = dig[f"{pan}_x"], dig[f"{pan}_sup"], dig[f"{pan}_inf"]    libre = (sup < 0.80) & (inf > -1.22)          # no recortado por el marco    ax.fill_between(xd[libre], inf[libre], sup[libre], alpha=.2,                    label="región estable (paper)")    xx = np.linspace(2.2, 20, 400)    ax.plot(xx, 1/(1 - xx/2), lw=2, label=r"$1/(1-r_0/2\ell)$  desplazado")    xx6 = np.linspace(6.2, 20, 400)    ax.plot(xx6, 1/(1 - xx6/6), lw=2, ls="--", label=r"$1/(1-r_0/6\ell)$  suavizado")    ax.set_xlim(0, 21); ax.set_ylim(-1.4, 0.9)    ax.axhline(0, color="k", lw=.8)    ax.set_xlabel(r"$r_0/\ell$"); ax.set_ylabel("$m/m_B$"); ax.set_title(tit)    ax.legend(fontsize=8)plt.tight_layout(); plt.show()

## Qué queda abiertoRegistrado en `docs/discrepancias.md`:- **D1** — a $r_0/\ell=3$, $m/m_B\approx2$, el código da relajación oscilatoria con  tasa $-0.72$ y período $2.44$. La Fig. 1(a) muestra curvas suaves sobre $s/r_0$  hasta **300**. Los bloques 1 y 3 coinciden entre sí al 0.17 % pero **ambos**  discrepan del paper, lo que descarta un error de integración temporal.- **D2** — a $m/m_B=-3.91$ el paper muestra comportamiento casi marginal; el código  da crecimiento hasta $10^6$ en $s=12$ (tramo convergido: dos resoluciones  coinciden a cuatro cifras).- **Descartado**: un reescalado de $\ell$ no explica D1 — ningún cutoff da una  relajación tan lenta.- **Pendiente**: completar el barrido R2 del regulador desplazado (ec. 16, mucho  más cara) y correr R1 sobre un rango de cutoffs.Los 40+ tests de `tests/` son estas mismas comprobaciones en forma ejecutable,con los valores analíticos esperados escritos al lado:`test_block1.py` ↔ secciones 1–2, `test_blocks23.py` ↔ 3–5, `test_block4.py` ↔ 6.